In [1]:
import pandas as pd
import ir_datasets
from src.data import DATA_DIR_INTERIM
from topic_gen.evaluate import QrelsEvaluator, CohenKappa, MeanAverageError, AreaUnderReceiver, ROSKendallTau, ROSRBO, binarize_qrels

from src.data import load_qrels_from_path
from src.utils import format_score

from topic_gen import logger
logger.setLevel("DEBUG")

# Prerequisite
#### Can we reproduce the qrels quality by Thomas et al. with open models?
Yes! Open models outperform the results on GPT 4.1

In [13]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_INTERIM / "robust-reference"
predictions, names, metadata = load_qrels_from_path(BASE_DIR)

# binarize qrels
predictions = [binarize_qrels(qrels) for qrels in predictions]

In [14]:
# Evaluate qrels
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=binarize_qrels(ir_datasets.load(
        "disks45/nocr/trec-robust-2004").qrels_iter()),
    measures=[CohenKappa(), MeanAverageError(), AreaUnderReceiver()],
    bootstrap=20,
    names=names)

[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 9/2942 qrels in references but not in predictions.


In [15]:
df = pd.DataFrame(res)
df["score"] = df.apply(format_score, axis=1)
df = df.pivot(index="name", columns="measure", values="score").reset_index()

df = df.merge(metadata, left_on="name", right_on="date")

df[["model", "prompt", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver
0,qwen3-30B-no-think,-DNA-zero-shot,0.75 ± 0.02,0.10 ± 0.01,0.90 ± 0.01
1,gpt-4.1,-DNA-zero-shot,0.64 ± 0.01,0.17 ± 0.01,0.81 ± 0.01


### Do LLM judges benefit from additional information in the topic?
The original TREC topics are judged by an LLM but one or two topic components are masked. For example, the prompt `-DNA-zero-shot-masked-title` only contains the **description** and **narrative** of the TREC topic. 

- Generally the differences are low
- Masking the title outperforms the full topic
- More context seems to be better

In [6]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_INTERIM / "qrels-robust-topics-masked"
predictions, names, metadata = load_qrels_from_path(BASE_DIR)

# binarize qrels
predictions = [binarize_qrels(qrels) for qrels in predictions]

In [7]:
# Evaluate qrels
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=binarize_qrels(ir_datasets.load(
        "disks45/nocr/trec-robust-2004").qrels_iter()),
    measures=[CohenKappa(), MeanAverageError(), AreaUnderReceiver()],
    bootstrap=20,
    names=names)

[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 2/2949 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 16/2935 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 4/2947 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 8/2943 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 10/2941 qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:345) Missing qrels: 14/2937 qrels in references but not in predictions.


In [12]:
df = pd.DataFrame(res)
df["score"] = df.apply(format_score, axis=1)
df = df.pivot(index="name", columns="measure", values="score").reset_index()
df = df.merge(metadata, left_on="name", right_on="date")

partial_annotation_prompts = [
    "-DNA-zero-shot", "-DNA-zero-shot-masked-title", "-DNA-zero-shot-masked-description", "-DNA-zero-shot-masked-narrative",
    "-DNA-zero-shot-masked-title-description", "-DNA-zero-shot-masked-title-narrative", "-DNA-zero-shot-masked-description-narrative"]
df["prompt"] = pd.Categorical(df["prompt"], partial_annotation_prompts)
df.sort_values(by=["model", "prompt"])[["model", "prompt", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver
5,qwen3-30B-no-think,-DNA-zero-shot-masked-title,0.77 ± 0.02,0.10 ± 0.01,0.90 ± 0.01
1,qwen3-30B-no-think,-DNA-zero-shot-masked-description,0.70 ± 0.02,0.12 ± 0.01,0.87 ± 0.01
2,qwen3-30B-no-think,-DNA-zero-shot-masked-narrative,0.71 ± 0.02,0.12 ± 0.01,0.88 ± 0.01
3,qwen3-30B-no-think,-DNA-zero-shot-masked-title-description,0.62 ± 0.03,0.16 ± 0.01,0.84 ± 0.01
4,qwen3-30B-no-think,-DNA-zero-shot-masked-title-narrative,0.71 ± 0.03,0.12 ± 0.01,0.87 ± 0.01
0,qwen3-30B-no-think,-DNA-zero-shot-masked-description-narrative,0.63 ± 0.02,0.15 ± 0.01,0.84 ± 0.01
